# 05 — Operational Reporting and Portfolio Outputs

This notebook creates the nonspatial tables and figures for:

- ArcGIS Dashboard design
- StoryMap creation
- README GitHub documentation
- Operational workload summaries

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arcpy

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TREES_PRIORITY = str(GDB / "trees_priority")

if not arcpy.Exists(TREES_PRIORITY):
    raise FileNotFoundError(f"Run Notebook 4. Missing: {TREES_PRIORITY}")

In [ ]:
candidate_fields = [ "ASSET_ID","PARK","COMMON","SPECIES","DBH_ORIG","DBH_CLEAN","DBH_IMPUTED","DBH_METHOD","MATURITY","AGE_PROXY",
                    "SIM_COND","LAST_INSP","PRIORITY","PRIOR_CLASS","ANNUAL_SEL","PLAN_YEAR","INSP_HRS","NEAR_SCHOOL","NEAR_SIDEWALK",
                    "NEAR_TRAIL","NEAR_ARTERIAL","EQUITY_HIGH","QA_NOTES","SPATIAL_NOTE","MODEL_VER","MODEL_NOTE",]

available = {field.name for field in arcpy.ListFields(TREES_PRIORITY)}
fields = [field for field in candidate_fields if field in available]

array = arcpy.da.TableToNumPyArray(TREES_PRIORITY, fields, skip_nulls=False, null_value=np.nan,)
df = pd.DataFrame(array)

print(f"Loaded {len(df):,} tree assets.")
df.head()

In [ ]:
# Dashboard KPIs
kpis = pd.DataFrame([{
    "total_tree_assets" : len(df),
    "year1_selected" : int(df["ANNUAL_SEL"].fillna(0).sum()),
    "year1_share_pct": round(df["ANNUAL_SEL"].fillna(0).mean()* 100, 2),
    "critical_assets" : int((df["PRIOR_CLASS"] == "Critical").sum()),
    "high_assets" : int((df["PRIOR_CLASS"] == "High").sum()),
    "high_equity_assets": int(df["EQUITY_HIGH"].fillna(0).sum()),
    "mean_priority_score" : round(df["PRIORITY"].mean(), 2,),
    "estimated_year1_hours" : round(df.loc[df["ANNUAL_SEL"] ==1, "INSP_HRS",].sum(), 1),
    "average_dbh_inches": round(df["DBH_CLEAN"].mean(), 1,),
    "estimated_dbh_values" : int(df["DBH_IMPUTED"].fillna(0).sum()),
    "dbh_imputed_pct" : round(df["DBH_IMPUTED"].fillna(0).mean() * 100, 2),
    "near_arterial_assets" : int(df["NEAR_ARTERIAL"].fillna(0).sum()),
    "avg_inspection_minutes" : round((df.loc[df["ANNUAL_SEL"] == 1, "INSP_HRS"].sum() / df["ANNUAL_SEL"].sum()) * 60, 1),
    
}])

kpis.to_csv(OUTPUT_DIR / "dashboard_kpis_revised.csv", index=False,)
kpis

In [ ]:
#Create a summary for the Priority dataset and export to csv.
priority_summary = (
    df.groupby("PRIOR_CLASS", dropna=False).agg(
        tree_count=("ASSET_ID", "count"),
        estimated_hours=("INSP_HRS", "sum"),
        avg_score=("PRIORITY", "mean")
    ).reset_index()
)
priority_summary.to_csv(OUTPUT_DIR / "priority_class_summary_revised.csv", index=False,)

#Create a summary for how manture trees are within study area and export to csv.
maturity_summary = (df.groupby("MATURITY",dropna=False).agg(
    tree_count=("ASSET_ID", "count"),
    year1_selected=("ANNUAL_SEL", "sum"),
    average_priority=("PRIORITY", "mean"),
    average_dbh=("DBH_CLEAN", "mean"),
).reset_index())

maturity_summary.to_csv(OUTPUT_DIR / "maturity_priority_summary.csv", index=False)

# Create DBH method summary and export to csv.
dbh_method_summary = (df.groupby(["DBH_IMPUTED", "DBH_METHOD"], dropna=False,).agg(
    tree_count=("ASSET_ID", "count"),
    average_priority=("PRIORITY", "mean"),
).reset_index())

dbh_method_summary.to_csv(OUTPUT_DIR / "dbh_method_priority_summary.csv", index=False,)

priority_summary

In [ ]:
if "PARK" in df.columns:
    park_summary = (
        df.groupby("PARK", dropna=False,).agg(
            tree_count=("ASSET_ID","count"),
            year1_selected=("ANNUAL_SEL", "sum"),
            average_priority=("PRIORITY", "mean"),
            estimates_hours=("INSP_HRS", "sum"),
            near_arterial=("NEAR_ARTERIAL", "sum")
        )
        .reset_index().sort_values(["year1_selected","average_priority"], ascending=False,)
    )
    park_summary.to_csv(OUTPUT_DIR / "park_workload_summary_revised.csv", index=False,)

if "COMMON" in df.columns and "SPECIES" in df.columns:
    species_summary =(
        df.groupby(["COMMON", "SPECIES"], dropna=False).agg(
            tree_count=("ASSET_ID", "count"),
            year1_selected=("ANNUAL_SEL", "sum"),
            average_priority=("PRIORITY", "mean"),
            average_dbh=("DBH_CLEAN", "mean"),
        ).reset_index().sort_values("tree_count", ascending=False,)
    )
    species_summary.to_csv(OUTPUT_DIR / "species_summary_revised.csv", index=False,)

annual_plan = (df.groupby("PLAN_YEAR", dropna=False,).agg(
    tree_count=("ASSET_ID", "count"),
    estimated_hours=("INSP_HRS", "sum"),
    average_priority=("PRIORITY", "mean")
).reset_index().sort_values("PLAN_YEAR"))

annual_plan.to_csv(OUTPUT_DIR / "annual_inspection_plan_revised.csv", index=False)

top_assets = (df.sort_values("PRIORITY", ascending=False).head(100))
top_assets.to_csv(OUTPUT_DIR / "top_100_inspection_candidates.csv", index=False,)

In [ ]:
# Figure 1: Priority Distribution

priority_order= ["Critical", "High", "Moderate", "Low",]
counts = (df["PRIOR_CLASS"].value_counts().reindex(priority_order).fillna(0))

fig, ax = plt.subplots(figsize=(8,5))
counts.plot(kind="bar", ax=ax)
ax.set_title("Seattle Tree Assets by Inspection Priority")
ax.set_xlabel("Priority Class")
ax.set_ylabel("Tree Assets")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

fig.savefig(FIGURE_DIR / "priority_class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: maturity profile
maturity_order = ["Young", "Established", "Mature", "Large Mature", "Unknown"]

maturity_counts = (df["MATURITY"].value_counts().reindex(maturity_order).fillna(0))

fig, ax = plt.subplots(figsize=(8,5))
maturity_counts.plot(kind="bar", ax=ax)
ax.set_title("DBH-Derived Maturity Classes")
ax.set_xlabel("Maturity Class")
ax.set_ylabel("Tree Assets")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

fig.savefig(FIGURE_DIR / "maturity_class_distribution.png", dpi=200, bbox_inches="tight",)
plt.show()

In [ ]:
# Figure 3: Parks with the most Year 1 candidates. 

if "park_summary" in globals():
    plot_df = (park_summary.dropna(subset=["PARK"]).head(12).sort_values("year1_selected"))

    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(plot_df["PARK"], plot_df["year1_selected"],)
    ax.set_title("Year 1 Inspection Candidates by Park")
    ax.set_xlabel("Selected Tree Assets")
    ax.set_ylabel("")
    fig.tight_layout

    fig.savefig(FIGURE_DIR / "year1_candidates_by_park.png", dpi=200, bbox_inches="tight",)

    plt.show()

In [ ]:
df.to_csv(OUTPUT_DIR / "tree_priority_analysis_table", index=False,)

print("Reporting outputs created:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(" -",path.relative_to(PROJECT_ROOT))

In [ ]:
year1_df = df[
    df["ANNUAL_SEL"] == 1
].copy()

largest_workload_by_park = (
    year1_df
    .groupby("PARK", dropna=False)
    .agg(
        year1_tree_count=("ASSET_ID", "count"),
        estimated_hours=("INSP_HRS", "sum"),
        average_priority=("PRIORITY", "mean"),
    )
    .reset_index()
    .sort_values(
        ["year1_tree_count", "estimated_hours"],
        ascending=False,
    )
)

display(
    largest_workload_by_park.head(10)
)